# Tratamento de dados (versão 2, do zero)

Percorremos **todas as colunas**, organizadas pelos módulos do Inquérito Nacional de Saúde (2019),
e para cada uma decidimos o tratamento.

Regras:
- treino e teste recebem **exatamente o mesmo tratamento**; tudo o que é "aprendido" dos dados
  (medianas, categorias, etc.) vem **só do treino**;
- a ordem das linhas do teste nunca muda (o `RowId` da submissão é a posição da linha);
- cada decisão fica registada com `registar(...)` → `resultados/decisoes_tratamento.csv`,
  que serve de base à secção de tratamento do relatório.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import warnings
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)  # avisos de desempenho ao criar colunas uma a uma

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

train = pd.read_csv("../dados/dataset_train.csv", low_memory=False)
test = pd.read_csv("../dados/dataset_test_X.csv", low_memory=False)
y = train["AGE_RANGE"]

# Limites das faixas (deduzidos dos dados: idades máximas declaradas em IN45 por faixa)
FAIXAS = ["15-17", "18-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49",
          "50-54", "55-59", "60-64", "65-69", "70-74", "75-79", "80-84", "85+"]

# X = treino sem o alvo + teste, tratados juntos; 'conjunto' permite separá-los no fim
X = pd.concat([train.drop(columns="AGE_RANGE"), test], keys=["treino", "teste"], names=["conjunto", "linha"])
print("treino", train.shape, "| teste", test.shape, "| X", X.shape)

treino (10962, 237) | teste (3655, 236) | X (14617, 236)


### Ferramentas de inspeção

- `ver(col)`: tipo, valores (treino e teste), **V de Cramér** com a faixa etária
  (0 = nenhuma associação, 1 = associação perfeita) e a faixa média por valor;
- `registar(...)`: guarda a decisão tomada para o relatório.

In [2]:
def v_cramer(a, b):
    t = pd.crosstab(a, b)
    if min(t.shape) < 2:
        return 0.0
    chi2 = chi2_contingency(t)[0]
    return np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1)))


def ver(col, max_valores=15):
    tr, te = train[col].astype(str), test[col].astype(str)
    print(f"=== {col}  ({train[col].dtype}) | {tr.nunique()} valores distintos | "
          f"V de Cramér com a idade = {v_cramer(tr, y):.3f}")
    tabela = pd.DataFrame({
        "treino": tr.value_counts(),
        "teste": te.value_counts(),
        "faixa_media": y.groupby(tr).mean().round(1),
    }).fillna(0).sort_values("treino", ascending=False)
    display(tabela.head(max_valores))
    if len(tabela) > max_valores:
        print(f"... mais {len(tabela) - max_valores} valores")
    so_teste = set(te) - set(tr)
    if so_teste:
        print("Valores que só aparecem no teste:", so_teste)


decisoes = []

def registar(colunas, modulo, significado, tratamento, porque):
    for c in ([colunas] if isinstance(colunas, str) else colunas):
        decisoes.append(dict(coluna=c, modulo=modulo, significado=significado,
                             tratamento=tratamento, justificacao=porque))

## 1. Identificação, data da entrevista, região e dados demográficos

`PID`, `HHID`, `PROXY`, `REFYEAR`, `REFMONTH`, `COUNTRY`, `NUTS1`, `NUTS2_AG`, `REGION`, `DEG_URB`, `SEX`,
`BIRTHPLACE_COD`, `CITIZEN_COD`, `BIRTHPLACEFATH_COD`, `BIRTHPLACEMOTH_COD`, `MARSTALEGAL`, `PARTNERS`

In [3]:
for c in ["PID", "HHID"]:
    print(c, "| únicos no treino:", train[c].is_unique, "| em comum treino/teste:", len(set(train[c]) & set(test[c])))
for c in ["PROXY", "REFYEAR", "REFMONTH", "COUNTRY"]:
    ver(c)

PID | únicos no treino: True | em comum treino/teste: 0
HHID | únicos no treino: True | em comum treino/teste: 0
=== PROXY  (float64) | 1 valores distintos | V de Cramér com a idade = 0.000


,treino,teste,faixa_media
PROXY,,,
1.0,10962,3655,8.8


=== REFYEAR  (int64) | 2 valores distintos | V de Cramér com a idade = 0.052


,treino,teste,faixa_media
REFYEAR,,,
2019,10745,3576,8.8
2020,217,79,7.7


=== REFMONTH  (int64) | 6 valores distintos | V de Cramér com a idade = 0.040


,treino,teste,faixa_media
REFMONTH,,,
10,4513,1443,8.9
11,3316,1125,8.8
12,1851,657,8.6
9,1065,351,8.7
1,214,78,7.7
2,3,1,5.0


=== COUNTRY  (object) | 1 valores distintos | V de Cramér com a idade = 0.000


,treino,teste,faixa_media
COUNTRY,,,
PT,10962,3655,8.8


In [4]:
print(pd.crosstab(train["REFYEAR"], train["REFMONTH"]))

REFMONTH   1   2     9     10    11    12
REFYEAR                                  
2019        0   0  1065  4513  3316  1851
2020      214   3     0     0     0     0


In [5]:
for c in ["NUTS1", "NUTS2_AG", "REGION", "DEG_URB"]:
    ver(c)
# REGION e NUTS2_AG não são a mesma divisão: o cruzamento mostra o que cada uma separa
print(pd.crosstab(train["REGION"], train["NUTS2_AG"]))

=== NUTS1  (object) | 3 valores distintos | V de Cramér com a idade = 0.087


,treino,teste,faixa_media
NUTS1,,,
Continente,8092,2642,9.0
RA Açores,1443,526,7.9
RA Madeira,1427,487,8.5


=== NUTS2_AG  (object) | 7 valores distintos | V de Cramér com a idade = 0.071

,treino,teste,faixa_media
NUTS2_AG,,,
Centro,1974,666,9.4
Lx V Tejo,1772,616,8.6
Norte,1625,496,8.7
RA Açores,1443,526,7.9
Alentejo,1439,463,9.4
RA Madeira,1427,487,8.5
Algarve,1282,401,8.7


=== REGION  (object) | 7 valores distintos | V de Cramér com a idade = 0.073

,treino,teste,faixa_media
REGION,,,
Centro (PT),2315,792,9.3
Alentejo,1848,593,9.4
Norte,1625,496,8.7
Região Autónoma dos Açores,1443,526,7.9
Região Autónoma da Madeira,1427,487,8.5
Algarve,1282,401,8.7
Lisboa,1022,360,8.2


=== DEG_URB  (object) | 3 valores distintos | V de Cramér com a idade = 0.103

,treino,teste,faixa_media
DEG_URB,,,
Áreas mediamente povoadas,4200,1426,8.4
Áreas pouco povoadas,3587,1179,9.4
Áreas densamente povoadas,3175,1050,8.5


NUTS2_AG                    Alentejo  Algarve  Centro  Lx V Tejo  Norte  RA Açores  RA Madeira
REGION                                                                                        
Alentejo                        1439        0       0        409      0          0           0
Algarve                            0     1282       0          0      0          0           0
Centro (PT)                        0        0    1974        341      0          0           0
Lisboa                             0        0       0       1022      0          0           0
Norte                              0        0       0          0   1625          0           0
Região Autónoma da Madeira         0        0       0          0      0          0        1427
Região Autónoma dos Açores         0        0       0          0      0       1443           0

In [6]:
for c in ["SEX", "BIRTHPLACE_COD", "CITIZEN_COD", "BIRTHPLACEFATH_COD", "BIRTHPLACEMOTH_COD"]:
    ver(c)

=== SEX  (object) | 2 valores distintos | V de Cramér com a idade = 0.076

,treino,teste,faixa_media
SEX,,,
Mulher,6254,2041,8.9
Homem,4708,1614,8.5


=== BIRTHPLACE_COD  (object) | 4 valores distintos | V de Cramér com a idade = 0.108

,treino,teste,faixa_media
BIRTHPLACE_COD,,,
Autóctone (PT),10151,3393,8.9
Outro país fora da UE,614,201,7.1
Outro estado membro da UE,184,60,7.1
Sem resposta,13,1,5.8


=== CITIZEN_COD  (object) | 4 valores distintos | V de Cramér com a idade = 0.089


,treino,teste,faixa_media
CITIZEN_COD,,,
Nacional (PT),10707,3571,8.8
Nacionalidade de outro país fora da UE,181,56,5.5
Nacionalidade de outro estado membro da UE,71,26,8.4
Sem resposta,3,2,5.0


=== BIRTHPLACEFATH_COD  (object) | 4 valores distintos | V de Cramér com a idade = 0.104

,treino,teste,faixa_media
BIRTHPLACEFATH_COD,,,
Autóctone (PT),10401,3460,8.9
Outro país fora da UE,414,152,5.9
Outro estado membro da UE,102,31,7.6
Sem resposta,45,12,8.7


=== BIRTHPLACEMOTH_COD  (object) | 4 valores distintos | V de Cramér com a idade = 0.098


,treino,teste,faixa_media
BIRTHPLACEMOTH_COD,,,
Autóctone (PT),10382,3449,8.9
Outro país fora da UE,453,164,6.2
Outro estado membro da UE,103,40,7.5
Sem resposta,24,2,7.2


In [7]:
for c in ["MARSTALEGAL", "PARTNERS"]:
    ver(c)
print(pd.crosstab(train["MARSTALEGAL"], train["PARTNERS"]))

=== MARSTALEGAL  (object) | 5 valores distintos | V de Cramér com a idade = 0.394

,treino,teste,faixa_media
MARSTALEGAL,,,
Casado,5359,1818,9.4
Solteiro,2793,931,5.2
Viúvo,1677,554,12.6
Divorciado,1094,345,8.6
Sem resposta,39,7,7.7


=== PARTNERS  (object) | 4 valores distintos | V de Cramér com a idade = 0.342

,treino,teste,faixa_media
PARTNERS,,,
Com relação de conjugalidade legal ou de facto,5741,1968,9.1
Não aplicável,3006,986,10.5
Sem relação de conjugalidade,2187,696,5.7
Sem resposta,28,5,6.2


PARTNERS      Com relação de conjugalidade legal ou de facto  Não aplicável  Sem relação de conjugalidade  Sem resposta
MARSTALEGAL                                                                                                            
Casado                                                  5032            242                            81             4
Divorciado                                               159            594                           335             6
Sem resposta                                               4             21                             8             6
Solteiro                                                 510            860                          1412            11
Viúvo                                                     36           1289                           351             1


### Decisões do bloco 1

**Remover `PID`, `HHID`, `PROXY` e `COUNTRY`:** os dois primeiros são identificadores (únicos, sem nenhum valor
em comum entre treino e teste, logo não generalizam); os dois últimos são constantes (um só valor), não distinguem ninguém.

In [8]:
X = X.drop(columns=["PID", "HHID", "PROXY", "COUNTRY"])
registar(["PID", "HHID"], "Identificação", "Identificador da pessoa / do agregado",
         "Removida", "Identificador único, sem valores comuns entre treino e teste: não generaliza")
registar("PROXY", "Identificação", "Entrevista por interposta pessoa", "Removida", "Constante (sempre 1)")
registar("COUNTRY", "Identificação", "País", "Removida", "Constante (sempre PT)")
print(X.shape)

(14617, 232)


**Remover `REFYEAR` e `REFMONTH`:** a data da entrevista (set/2019 a fev/2020) não se relaciona com a idade
(qui-quadrado p = 0,15). Não temos data de nascimento, só a faixa (que é o alvo).

In [9]:
meses = (train["REFYEAR"] - 2019) * 12 + train["REFMONTH"]
print("faixa média por mês de entrevista:", y.groupby(meses).mean().round(2).to_dict())
print("p-valor do qui-quadrado:", round(chi2_contingency(pd.crosstab(meses, y))[1], 3))
X = X.drop(columns=["REFYEAR", "REFMONTH"])
registar(["REFYEAR", "REFMONTH"], "Identificação", "Ano / mês da entrevista", "Removida",
         "Sem associação significativa com a idade (qui-quadrado p = 0,15)")

faixa média por mês de entrevista: {9: 8.67, 10: 8.86, 11: 8.8, 12: 8.6, 13: 7.73, 14: 5.0}
p-valor do qui-quadrado: 0.153


**Região: manter só `REGION`.** `NUTS1` é uma agregação de `REGION` (Continente / Açores / Madeira, que já são
regiões em `REGION`). `NUTS2_AG` usa outra divisão ("Lx V Tejo" junta Lisboa com partes do Alentejo e do Centro),
com associação à idade semelhante. `REGION` é a divisão oficial e a mais simples de explicar.

In [10]:
for c in ["NUTS1", "NUTS2_AG", "REGION"]:
    print(f"{c:9s} V de Cramér = {v_cramer(train[c], y):.3f}")
X = X.drop(columns=["NUTS1", "NUTS2_AG"])
X["REGION"] = X["REGION"].astype("category")
registar("REGION", "Região", "Região NUTS II", "Categórica nominal", "Divisão oficial; contém a informação de NUTS1")
registar(["NUTS1", "NUTS2_AG"], "Região", "Região NUTS I / NUTS II agregada", "Removida",
         "Redundante com REGION (NUTS1 é agregação; NUTS2_AG é outra divisão com a mesma informação útil)")

NUTS1     V de Cramér = 0.087
NUTS2_AG  V de Cramér = 0.071
REGION    V de Cramér = 0.073


**`DEG_URB` → ordinal** (0 = pouco, 1 = medianamente, 2 = densamente povoada). Não tem valores em falta
nem no treino nem no teste, por isso não é preciso preencher nada.
**`SEX` → binária** (1 = Mulher).

In [11]:
print("DEG_URB em falta: treino", train["DEG_URB"].isna().sum(), "| teste", test["DEG_URB"].isna().sum())
X["DEG_URB"] = X["DEG_URB"].map({"Áreas pouco povoadas": 0, "Áreas mediamente povoadas": 1, "Áreas densamente povoadas": 2})
X["SEX"] = (X["SEX"] == "Mulher").astype(int)
assert X[["DEG_URB", "SEX"]].notna().all().all()
registar("DEG_URB", "Região", "Grau de urbanização", "Ordinal 0-2", "Ordem natural (pouco < mediamente < densamente povoada)")
registar("SEX", "Demografia", "Sexo", "Binária (1 = Mulher)", "Duas categorias, sem valores em falta")

DEG_URB em falta: treino 0 | teste 0


**Local de nascimento e nacionalidade.**
- `BIRTHPLACE_COD` e `CITIZEN_COD` mantêm-se: juntas distinguem, por exemplo, os nascidos fora da UE com
  nacionalidade portuguesa (faixa média 7,8, provavelmente retornados) dos estrangeiros (5,5, imigração recente).
- Pai e mãe têm o mesmo valor em 97% dos casos. Testámos substituí-los por **`PAIS_NASCIDOS_FORA`** = quantos
  dos pais nasceram fora de Portugal (0 / 1 / 2): a associação com a idade é maior (V 0,124 vs 0,10), mas
  **a previsão piorou** (HGB, 5 folds × 3 sementes: 0,4767 → 0,4727, pior em 4 dos 5 folds) — a distinção
  UE / fora da UE dos pais perde-se na contagem. **Decisão: manter as duas colunas originais.**

In [12]:
g = pd.DataFrame({"nascimento x nacionalidade": train["BIRTHPLACE_COD"] + " / " + train["CITIZEN_COD"]})
display(g.assign(faixa=y).groupby("nascimento x nacionalidade")["faixa"].agg(["size", "mean"]).round(1))


def fora(col):
    # 1 = nasceu fora de PT, 0 = nasceu em PT, NaN = sem resposta
    return X[col].map(lambda v: np.nan if v == "Sem resposta" else float(v != "Autóctone (PT)"))

pais_fora = fora("BIRTHPLACEFATH_COD") + fora("BIRTHPLACEMOTH_COD")   # só para análise; não fica nos dados
print(pais_fora.value_counts(dropna=False))
print("V de Cramér:", round(v_cramer(pais_fora.loc["treino"].fillna(-1).values, y.values), 3),
      "| pai:", round(v_cramer(train["BIRTHPLACEFATH_COD"], y), 3), "| mãe:", round(v_cramer(train["BIRTHPLACEMOTH_COD"], y), 3))

for c in ["BIRTHPLACE_COD", "CITIZEN_COD", "BIRTHPLACEFATH_COD", "BIRTHPLACEMOTH_COD"]:
    X[c] = X[c].astype("category")
registar(["BIRTHPLACE_COD", "CITIZEN_COD"], "Demografia", "Local de nascimento / nacionalidade",
         "Categórica nominal ('Sem resposta' como categoria)", "Juntas distinguem retornados de imigrantes recentes")
registar(["BIRTHPLACEFATH_COD", "BIRTHPLACEMOTH_COD"], "Demografia", "Local de nascimento do pai / mãe",
         "Categórica nominal", "Testada a contagem PAIS_NASCIDOS_FORA: piorou a previsão (-0,004, pior em 4/5 folds)")

,size,mean
nascimento x nacionalidade,,
Autóctone (PT) / Nacional (PT),10151,8.9
Outro estado membro da UE / Nacional (PT),116,6.3
Outro estado membro da UE / Nacionalidade de outro estado membro da UE,67,8.4
Outro estado membro da UE / Nacionalidade de outro país fora da UE,1,6.0
Outro país fora da UE / Nacional (PT),429,7.8
Outro país fora da UE / Nacionalidade de outro estado membro da UE,4,7.5
Outro país fora da UE / Nacionalidade de outro país fora da UE,180,5.5
Outro país fora da UE / Sem resposta,1,9.0
Sem resposta / Nacional (PT),11,6.4


0.0    13637
2.0      535
1.0      383
NaN       62
Name: count, dtype: int64


V de Cramér: 0.124 | pai: 0.104 | mãe: 0.098


**Estado civil e conjugalidade.**
- `MARSTALEGAL` (V = 0,39): categórica nominal; "Sem resposta" (39 casos) fica como categoria — a distribuição
  de idades não coincide com nenhuma das outras respostas.
- `PARTNERS`: "Não aplicável" corresponde **exatamente** a quem vive sozinho (`HHNBPERS` = 1) → renomeado
  "Vive sozinho". Não é o mesmo que "Sem relação de conjugalidade" (vive com outras pessoas, sem companheiro):
  faixas médias 10,5 vs 5,7.

In [13]:
print(pd.crosstab(train["PARTNERS"], train["HHNBPERS_COD"]))
print(y.groupby(train["PARTNERS"]).mean().round(1))
X["PARTNERS"] = X["PARTNERS"].replace({"Não aplicável": "Vive sozinho"})
for c in ["MARSTALEGAL", "PARTNERS"]:
    X[c] = X[c].astype("category")
registar("MARSTALEGAL", "Demografia", "Estado civil legal", "Categórica nominal ('Sem resposta' como categoria)",
         "Sem ordem natural; 'Sem resposta' não se parece com nenhuma outra resposta")
registar("PARTNERS", "Demografia", "Vive com companheiro/a", "Categórica; 'Não aplicável' → 'Vive sozinho'",
         "'Não aplicável' coincide exatamente com agregados de 1 pessoa")
print(X.shape)

HHNBPERS_COD                                     1.0   2.0   3.0  4.0  5.0  6.0  >=7
PARTNERS                                                                            
Com relação de conjugalidade legal ou de facto     0  3126  1356  969  219   44   27
Não aplicável                                   3006     0     0    0    0    0    0
Sem relação de conjugalidade                       0   882   711  427  125   30   12
Sem resposta                                       0    13     6    7    0    1    1
PARTNERS
Com relação de conjugalidade legal ou de facto     9.1
Não aplicável                                     10.5
Sem relação de conjugalidade                       5.7
Sem resposta                                       6.2
Name: AGE_RANGE, dtype: float64
(14617, 228)


## 2. Agregado familiar

`HHNBPERS_COD` (nº de pessoas), `HHNBPERS_0_13_COD` (nº de crianças 0-13), `HH_ACT` / `HH_INACT`
(nº de pessoas **em idade ativa** que trabalham / não trabalham), `HHTYPE` (tipo de agregado).

In [14]:
for c in ["HHNBPERS_COD", "HHNBPERS_0_13_COD", "HH_ACT", "HH_INACT", "HHTYPE"]:
    ver(c)

=== HHNBPERS_COD  (object) | 7 valores distintos | V de Cramér com a idade = 0.230


,treino,teste,faixa_media
HHNBPERS_COD,,,
2.0,4021,1368,9.9
1.0,3006,986,10.5
3.0,2073,690,6.8
4.0,1403,467,5.7
5.0,344,110,5.8
6.0,75,23,5.9
>=7,40,11,6.0


=== HHNBPERS_0_13_COD  (object) | 4 valores distintos | V de Cramér com a idade = 0.324


,treino,teste,faixa_media
HHNBPERS_0_13_COD,,,
0.0,9087,3034,9.4
1.0,1236,406,5.5
2.0,551,183,5.5
>=3,88,32,5.4


=== HH_ACT  (float64) | 7 valores distintos | V de Cramér com a idade = 0.318


,treino,teste,faixa_media
HH_ACT,,,
0.0,4691,1575.0,11.5
1.0,2925,935.0,7.4
2.0,2811,970.0,6.1
3.0,451,150.0,6.0
4.0,73,24.0,5.5
5.0,8,1.0,4.1
6.0,3,0.0,6.7


=== HH_INACT  (float64) | 6 valores distintos | V de Cramér com a idade = 0.220


,treino,teste,faixa_media
HH_INACT,,,
0.0,7091.0,2344.0,9.6
1.0,2779.0,954.0,7.5
2.0,918.0,289.0,6.6
3.0,142.0,57.0,5.9
4.0,25.0,10.0,6.2
5.0,7.0,0.0,4.7
6.0,0.0,1.0,0.0


Valores que só aparecem no teste: {'6.0'}
=== HHTYPE  (object) | 7 valores distintos | V de Cramér com a idade = 0.289


,treino,teste,faixa_media
HHTYPE,,,
Casal sem filhos,3126,1087,10.5
Agregado unipessoal,3006,986,10.5
Casal com pelo menos um filho < 25 anos,2396,795,5.4
Outro tipo de agregado,757,256,8.0
Casal com todos os filhos >= 25 anos,690,239,8.8
Pai ou mãe com pelo menos um filho < 25 anos,591,168,4.8
Pai ou mãe com todos os filhos >= 25 anos,396,124,9.2


**Quem entra em `HH_ACT` / `HH_INACT`?** Entre quem vive sozinho, `HH_ACT` = `HH_INACT` = 0 acontece
(praticamente) só a partir dos 65 anos: as pessoas com 65 ou mais não são contadas nestas duas colunas.

In [15]:
sozinhos = train["HHNBPERS_COD"] == "1.0"
print(pd.crosstab([train.loc[sozinhos, "HH_ACT"], train.loc[sozinhos, "HH_INACT"]], y[sozinhos]))

AGE_RANGE        0   1   2   3   4   5    6    7    8    9    10   11   12   13   14   15
HH_ACT HH_INACT                                                                          
0.0    0.0        1   0   0   0   0   0    0    0    0    0    0  314  355  355  312  290
       1.0        1   4  14   8   6   9   20   35   46  104  174    0    0    0    0    0
1.0    0.0        0   0  25  54  64  68  117  144  165  183  138    0    0    0    0    0

**Tratamento:**
- `HHNBPERS_COD` e `HHNBPERS_0_13_COD` → numéricas (os limites "≥7" e "≥3" passam a 7 e 3);
- `HHTYPE` → categórica nominal;
- nova **`N_FORA_IDADE_ATIVA`** = pessoas − ativos − inativos − crianças 0-13: nº de membros fora da idade ativa
  (65 ou mais, e jovens de 14-15). Calculada linha a linha só com colunas que existem no treino e no teste
  (não usa o alvo). Os poucos valores negativos (causados pelos limites "≥7"/"≥3") passam a 0.

In [16]:
X["HHNBPERS_COD"] = X["HHNBPERS_COD"].replace(">=7", "7").astype(float)
X["HHNBPERS_0_13_COD"] = X["HHNBPERS_0_13_COD"].replace(">=3", "3").astype(float)
X["HHTYPE"] = X["HHTYPE"].astype("category")

resto = X["HHNBPERS_COD"] - X["HH_ACT"] - X["HH_INACT"] - X["HHNBPERS_0_13_COD"]
print("valores negativos (passam a 0):", (resto < 0).sum())
X["N_FORA_IDADE_ATIVA"] = resto.clip(lower=0)
print(y.groupby(X.loc["treino", "N_FORA_IDADE_ATIVA"].values).agg(["size", "mean"]).round(1))

registar(["HHNBPERS_COD", "HHNBPERS_0_13_COD"], "Agregado", "Nº de pessoas / de crianças 0-13 no agregado",
         "Numérica ('≥7' → 7, '≥3' → 3)", "Contagens: a ordem importa")
registar(["HH_ACT", "HH_INACT"], "Agregado", "Nº de pessoas em idade ativa que trabalham / não trabalham",
         "Numérica (sem alteração)", "Já são contagens; quem tem 65+ não é contado")
registar("HHTYPE", "Agregado", "Tipo de agregado", "Categórica nominal", "Sem ordem natural")
registar("N_FORA_IDADE_ATIVA", "Agregado", "Nº de membros fora da idade ativa (nova)",
         "Numérica = pessoas − ativos − inativos − crianças, mínimo 0",
         "Subtração que as árvores não fazem sozinhas; separa agregados com idosos")
print(X.shape)

valores negativos (passam a 0):

 8
     size  mean
0.0  5667   6.5
1.0  3332  10.6
2.0  1914  12.0
3.0    47  11.2
4.0     2  13.0
(14617, 229)


## 3. Colunas `IN` (perguntas nacionais: seguros, dificuldades, saúde oral, trabalho, saúde reprodutiva, tabaco, opiniões, mobilidade)

### Regra geral para as respostas especiais
- **"Não aplicável"** quase sempre resulta de uma **pergunta-filtro** anterior (ex.: `IN7a` só é perguntado a quem
  respondeu "Sim" em `IN6`). Verificamos isso abaixo com cruzamentos. Nesse caso o "Não aplicável" passa a `NaN`
  **sem perder informação**: o motivo continua guardado na pergunta-filtro, e os modelos de árvores tratam o `NaN`
  como um valor próprio (o HGB envia-o para um ramo; o RF recebe-o como -1).
- **"Sem resposta"** e **"NR"** são a mesma coisa (unificados) e passam a `NaN`. Se a coluna também tiver
  "Não aplicável" e o "Sem resposta" for ≥ 1% das linhas, cria-se a indicadora `<COLUNA>_SEM_RESP` para não
  confundir "não se perguntou" com "não respondeu".

In [17]:
ESPECIAIS = {"Não aplicável", "Sem resposta", "NR", "nan"}


def cruzar(a, b):
    display(pd.crosstab(X[a].astype(str).str[:30], X[b].astype(str).str[:30]))


def sem_resposta(c):
    s = X[c].astype(str)
    sr = s.isin(["Sem resposta", "NR"])
    if (s == "Não aplicável").any() and sr.mean() >= 0.01:
        X[c + "_SEM_RESP"] = sr.astype(int)
        return f" + indicadora {c}_SEM_RESP"
    return ""


def ordinal(cols, mapa):
    """Aplica a escala 'mapa'; especiais → NaN. Falha se aparecer um valor não previsto."""
    for c in cols:
        extra = sem_resposta(c)
        s = X[c].astype(str)
        imprevistos = set(s) - set(mapa) - ESPECIAIS
        assert not imprevistos, (c, imprevistos)
        X[c] = s.map(mapa).astype(float)
    return extra


def numerica(cols, limites=None):
    """Texto numérico → número; 'limites' converte valores como '53 ou mais' → 53."""
    for c in cols:
        sem_resposta(c)
        s = X[c].astype(str).replace(limites or {})
        valor = pd.to_numeric(s, errors="coerce")
        imprevistos = set(s[valor.isna()]) - ESPECIAIS
        assert not imprevistos, (c, imprevistos)
        X[c] = valor


def categorica(cols):
    for c in cols:
        sem_resposta(c)
        X[c] = X[c].astype(str).replace({"NR": "Sem resposta", "Não aplicável": np.nan, "nan": np.nan}).astype("category")


SIM_NAO = {"Sim": 1, "Não": 0}
DIFICULDADE = {"Nenhuma dificuldade": 0, "Alguma dificuldade": 1, "Muita dificuldade": 2, "Não consegue de todo": 3}

### 3.1 Seguros / subsistemas de saúde: `IN1`, `IN5`, `IN6`, `IN7a`–`IN7h`
`IN7a`–`IN7h` só são perguntados a quem respondeu "Sim" em `IN6` (confirmado abaixo) → "Não aplicável" = `NaN`.

In [18]:
cruzar("IN7a", "IN6")
for c in ["IN1", "IN5", "IN6"]:
    ver(c, 8)

IN6,Não,Sem resposta,Sim
IN7a,,,
Não,0,0,1723
Não aplicável,11499,100,0
Sem resposta,0,0,257
Sim,0,0,1038


=== IN1  (object) | 4 valores distintos | V de Cramér com a idade = 0.152


,treino,teste,faixa_media
IN1,,,
Sim,8678,2910,8.6
Não,1467,481,10.5
Não aplicável,806,261,7.1
Sem resposta,11,3,8.5


=== IN5  (object) | 8 valores distintos | V de Cramér com a idade = 0.067


,treino,teste,faixa_media
IN5,,,
Não aplicável,8810,2985,8.7
ADSE,1526,491,8.9
Outro,205,52,9.4
SAMS,176,58,8.7
ADM,117,27,9.0
SAD/GNR,71,22,8.5
SAD/PSP,53,16,8.3
Sem resposta,4,4,5.8


=== IN6  (object) | 3 valores distintos | V de Cramér com a idade = 0.180


,treino,teste,faixa_media
IN6,,,
Não,8645,2854,9.2
Sim,2244,774,7.2
Sem resposta,73,27,6.9


In [19]:
IN7 = [f"IN7{l}" for l in "abcdefgh"]
ordinal(["IN1", "IN6"] + IN7, SIM_NAO)
categorica(["IN5"])
registar(["IN1", "IN6"] + IN7, "IN: seguros", "Perguntas Sim/Não sobre seguros e subsistemas", "Binária 1/0; especiais → NaN",
         "IN7* só perguntado se IN6 = Sim: o 'Não aplicável' está explicado por IN6")
registar("IN5", "IN: seguros", "Subsistema de saúde (ADSE, SAMS, ...)", "Categórica nominal", "Sem ordem natural")

### 3.2 Dificuldades: `IN8`, `IN9`, `IN15`
Escala ordinal 0 (nenhuma) – 3 (não consegue). `IN15` só é perguntado a partir dos 55 anos (o "Não aplicável"
é um filtro de idade): fica `NaN`, que o modelo distingue de qualquer resposta — é uma das colunas mais informativas.

In [20]:
print(pd.crosstab(X.loc["treino", "IN15"].eq("Não aplicável").values, y.values))
ordinal(["IN8", "IN9", "IN15"], DIFICULDADE)
registar(["IN8", "IN9", "IN15"], "IN: dificuldades", "Grau de dificuldade", "Ordinal 0-3; especiais → NaN",
         "Ordem natural; em IN15 o 'Não aplicável' (NaN) identifica quem tem menos de 55 anos")

col_0   0    1    2    3    4    5    6    7    8     9     10    11    12   13   14   15
row_0                                                                                    
False    0    0    0    0    0    0    0    0    0  1014  1062  1040  1000  823  650  496
True   207  172  399  377  449  647  859  862  905     0     0     0     0    0    0    0

### 3.3 Idades / anos declarados: `IN2`, `IN3`, `IN4`
Números guardados como texto (idades e durações). Convertidos para número.

In [21]:
numerica(["IN2", "IN3", "IN4"])
registar(["IN2", "IN3", "IN4"], "IN: idades", "Idades / anos declarados", "Numérica; especiais → NaN", "Já são quantidades")

### 3.4 Saúde oral: `IN16`, `IN17`, `IN18`, `IN19`
- `IN16` (motivo da última consulta no dentista) está vazio exatamente quando `AM1` diz que a última consulta foi
  há 12 meses ou mais / nunca → vazio = `NaN` explicado por `AM1`.
- `IN17` (porque não foi ao dentista): "Não aplicável" = quem foi.
- `IN18` (frequência de lavagem dos dentes) e `IN19` (há quanto tempo …) → ordinais; "NR" = "Sem resposta".

In [22]:
cruzar("IN16", "AM1")
categorica(["IN16", "IN17"])
ordinal(["IN18"], {"Nunca": 0, "Menos de uma vez por semana": 1, "Algumas vezes por semana": 2,
                   "Todos os dias, apenas de manhã": 3, "Todos os dias, apenas antes de se deitar": 3,
                   "Todos os dias, de manhã e antes de se deitar": 4,
                   "Todos os dias, de manhã, após o almoço e antes de se deitar": 5})
ordinal(["IN19"], {"Há menos de 10 anos": 0, "Entre 10 anos e menos de 20 anos": 1, "20 anos ou mais": 2, "Nunca": 3})
registar(["IN16", "IN17"], "IN: saúde oral", "Motivo da consulta / de não ir ao dentista", "Categórica nominal",
         "Vazio/'Não aplicável' explicado por AM1")
registar("IN18", "IN: saúde oral", "Frequência de lavagem dos dentes", "Ordinal 0-5", "Ordem natural de frequência")
registar("IN19", "IN: saúde oral", "Há quanto tempo (escalões)", "Ordinal 0-3 (Nunca = 3); NR → NaN", "Ordem natural do tempo")

AM1,6 a menos de 12 meses,Há 12 meses ou mais,Há menos de 6 meses,Nunca,Sem resposta
IN16,,,,,
Aplicação de selantes de fissu,28,0,83,0,0
Extração de um dente,304,0,479,0,0
Fazer uma prótese dentária fix,200,0,409,0,0
Higienização,705,0,993,0,0
Ortodontia,78,0,296,0,0
Outra razão,46,0,65,0,0
Sem resposta,8,0,15,0,0
Sentiu dores ou outra situação,204,0,323,0,0
Tratamento de um dente,365,0,809,0,0


### 3.5 Trabalho: `IN60`–`IN66`
`IN60` só é perguntado a quem trabalha (confirmado com `MAINSTAT`). `IN62` (tipo de problema de saúde) tem 99% de
"Não aplicável" e quase nenhuma associação com a idade → removida.

In [23]:
cruzar("IN60", "MAINSTAT")
ver("IN62", 10)
ordinal(["IN60", "IN61", "IN63", "IN64", "IN65", "IN66"], SIM_NAO)
X = X.drop(columns="IN62")
registar(["IN60", "IN61", "IN63", "IN64", "IN65", "IN66"], "IN: trabalho", "Perguntas Sim/Não sobre trabalho e saúde",
         "Binária 1/0; especiais → NaN", "IN60 só perguntado a quem trabalha (explicado por MAINSTAT)")
registar("IN62", "IN: trabalho", "Tipo de problema de saúde ligado ao trabalho", "Removida",
         "99% 'Não aplicável' e V de Cramér 0,04")

MAINSTAT,"Aluno, estudante, ou em estági",Desempregado,Incapacitado permanente para o,Ocupa-se de tarefas domésticas,Outra situação de inatividade,Presta serviço cívico ou comun,"Reformado, com reforma antecip",Sem resposta,Tem um emprego ou trabalho (in
IN60,,,,,,,,,
Não,0,0,0,0,0,0,0,0,2636
Não aplicável,699,1004,364,757,141,1,5148,33,0
Sem resposta,0,0,0,0,0,0,0,0,35
Sim,0,0,0,0,0,0,0,0,3799


=== IN62  (object) | 8 valores distintos | V de Cramér com a idade = 0.042


,treino,teste,faixa_media
IN62,,,
Não aplicável,10852.0,3611.0,8.8
"Problemas ósseos, articulares ou musculares",73.0,28.0,8.4
Outros problemas de saúde,13.0,5.0,8.5
Problemas respiratórios ou pulmonares,9.0,2.0,11.0
"Stress, depressão ou ansiedade",6.0,5.0,5.7
Problemas de audição,5.0,1.0,9.8
Problemas de pele,3.0,0.0,7.7
Problemas de rins ou bexiga,1.0,2.0,10.0
"Problemas do aparelho digestivo (estômago, fígado, pâncreas ou intestino)",0.0,1.0,0.0


Valores que só aparecem no teste: {'Problemas do aparelho digestivo (estômago, fígado, pâncreas ou intestino)'}


### 3.6 Saúde reprodutiva (só mulheres): `IN20`–`IN28`
`IN20` só é perguntado a mulheres (em idade fértil); `IN21` (método) só a quem utiliza contraceção; `IN22`
(motivo) só a quem não utiliza. `IN24` é um **ano** → convertido em **anos desde** esse ano até à entrevista;
"Gravidez sem termo" fica numa indicadora. `IN25`/`IN26` têm limites ("53 ou mais", "33 ou mais").

In [24]:
cruzar("IN20", "SEX")
cruzar("IN21", "IN20")
cruzar("IN22", "IN20")

SEX,0,1
IN20,,
Não aplicável,6322,4633
Não utiliza,0,1762
Sem resposta,0,130
Utiliza,0,1770


IN20,Não aplicável,Não utiliza,Sem resposta,Utiliza
IN21,,,,
Abstinência periódica: calendá,0,0,0,1
Adesivo contracetivo,0,0,0,7
Anel,0,0,0,34
Coito interrompido,0,0,0,15
DIU (Dispositivo intrauterino),0,0,0,194
Diafragma,0,0,0,2
Implante,0,0,0,87
Laqueação de trompas ou Vasect,0,0,0,57
Não aplicável,10955,1762,130,0


IN20,Não aplicável,Não utiliza,Sem resposta,Utiliza
IN22,,,,
Está a amamentar,0,11,0,0
Está na menopausa,0,328,0,0
Não aplicável,10955,0,130,1770
Não tem atividade sexual atual,0,613,0,0
Outro motivo,0,575,0,0
Por razões de saúde,0,83,0,0
Quer engravidar,0,73,0,0
Sem resposta,0,79,0,0


In [25]:
ordinal(["IN20"], {"Não utiliza": 0, "Utiliza": 1})
ordinal(["IN23", "IN28"], SIM_NAO)
categorica(["IN21", "IN22"])

ano_entrevista = pd.concat([train["REFYEAR"], test["REFYEAR"]]).values
X["IN24_GRAVIDEZ_SEM_TERMO"] = (X["IN24"] == "Gravidez sem termo (n.a.)").astype(int)
numerica(["IN24"], {"Gravidez sem termo (n.a.)": "nan"})
X["IN24"] = ano_entrevista - X["IN24"]
numerica(["IN25"], {"53 ou mais": "53"})
numerica(["IN26"], {"33 ou mais": "33"})
ordinal(["IN27"], {"Não fumou": 0, "Ocasionalmente (menos de uma vez por mês)": 1, "Ocasionalmente (uma vez por mês)": 2,
                   "Ocasionalmente (alguns dias por mês)": 3, "Ocasionalmente (alguns dias por semana)": 4, "Diariamente": 5})
print(X["IN24"].describe())
registar(["IN20", "IN23", "IN28"], "IN: saúde reprodutiva", "Perguntas Sim/Não (contraceção, ...)", "Binária 1/0; especiais → NaN",
         "Só perguntado a mulheres: o NaN coincide com SEX / idade")
registar(["IN21", "IN22"], "IN: saúde reprodutiva", "Método contracetivo / motivo de não usar", "Categórica nominal",
         "'Não aplicável' explicado por IN20")
registar("IN24", "IN: saúde reprodutiva", "Ano (de um acontecimento reprodutivo)", "Anos desde esse ano até à entrevista",
         "Um ano em si não é comparável; a distância à entrevista é")
registar("IN24_GRAVIDEZ_SEM_TERMO", "IN: saúde reprodutiva", "Resposta especial de IN24 (nova)", "Indicadora 0/1",
         "Não é um ano: guardada à parte para não se perder")
registar(["IN25", "IN26"], "IN: saúde reprodutiva", "Quantidades (semanas / meses)", "Numérica ('53 ou mais' → 53, '33 ou mais' → 33)",
         "Limite superior tratado como o próprio limite")
registar("IN27", "IN: saúde reprodutiva", "Fumou durante a gravidez", "Ordinal 0-5", "Ordem natural de frequência")

count    1949.000000
mean       13.558235
std         9.187546
min         0.000000
25%         6.000000
50%        13.000000
75%        20.000000
max        50.000000
Name: IN24, dtype: float64


### 3.7 Comportamentos e alimentação: `IN29`–`IN42`, `IN68`–`IN71`

In [26]:
numerica(["IN29"])
IN30_42 = [f"IN{i}" for i in range(30, 43)]
ordinal(IN30_42, SIM_NAO)
FREQ = {"Nunca": 0, "Menos de uma vez por semana": 1, "1 a 3 vezes por semana": 2, "4 a 6 vezes por semana": 3,
        "Uma vez ou mais por dia": 4}
ordinal(["IN68", "IN69", "IN70", "IN71"], FREQ)
registar("IN29", "IN: comportamentos", "Valor numérico 1-3", "Numérica", "Já é número")
registar(IN30_42, "IN: comportamentos", "Perguntas Sim/Não", "Binária 1/0", "Duas respostas")
registar(["IN68", "IN69", "IN70", "IN71"], "IN: alimentação", "Frequência de consumo", "Ordinal 0-4", "Ordem natural de frequência")

### 3.8 Tabaco: `IN43`, `IN44`, `IN45`, `IN461`–`IN467`, `IN47`, `IN49`
`IN43` (fuma?) é o filtro de todas as outras: `IN44`/`IN47` só para quem fuma, `IN45`/`IN461`–`IN467` só para quem
já fumou (confirmado abaixo). `IN45` e `IN47` são idades.

In [27]:
cruzar("IN44", "IN43")
print(pd.crosstab(X["IN45"].eq("Não aplicável"), X["IN43"]))
print(pd.crosstab(X["IN47"].eq("Não aplicável"), X["IN43"]))

IN43,Fuma diariamente,Fuma ocasionalmente,"Já fumou, mas atualmente não f",Não fuma,Sem resposta
IN44,,,,,
Charutos,5,4,0,0,0
Cigarrilhas,36,7,0,0,0
Cigarros em maço,1741,263,0,0,0
Cigarros enrolados à mão,159,18,0,0,0
Não aplicável,0,0,3159,9036,52
Outro tipo,12,3,0,0,0
Sem resposta,3,0,0,0,0
Tabaco aquecido,107,6,0,0,0
Tabaco de cachimbo,3,3,0,0,0


IN43   Fuma diariamente  Fuma ocasionalmente  Já fumou, mas atualmente não fuma  Não fuma  Sem resposta
IN45                                                                                                   
False                 0                    0                               3159         0             0
True               2066                  304                                  0      9036            52
IN43   Fuma diariamente  Fuma ocasionalmente  Já fumou, mas atualmente não fuma  Não fuma  Sem resposta
IN47                                                                                                   
False              2066                  158                               2356         0             0
True                  0                  146                                803      9036            52


In [28]:
ordinal(["IN43"], {"Não fuma": 0, "Já fumou, mas atualmente não fuma": 1, "Fuma ocasionalmente": 2, "Fuma diariamente": 3})
categorica(["IN44", "IN49"])
numerica(["IN45", "IN47"])
IN46 = [f"IN46{i}" for i in range(1, 8)]
ordinal(IN46, SIM_NAO)
registar("IN43", "IN: tabaco", "Hábito tabágico", "Ordinal 0-3 (não fuma < já fumou < ocasional < diário)", "Ordem de exposição")
registar(["IN44", "IN49"], "IN: tabaco", "Tipo de tabaco / local onde fuma", "Categórica nominal", "'Não aplicável' explicado por IN43")
registar(["IN45", "IN47"], "IN: tabaco", "Idades declaradas (tabaco)", "Numérica", "São idades; 'Não aplicável' explicado por IN43")
registar(IN46, "IN: tabaco", "Perguntas Sim/Não para ex-fumadores", "Binária 1/0", "'Não aplicável' explicado por IN43")

### 3.9 Opiniões: `IN50`–`IN54` (escala de concordância de 7 pontos)

In [29]:
CONCORDANCIA = {"Totalmente em desacordo": 0, "Em desacordo": 1, "Mais ou menos em desacordo": 2,
                "Nem de acordo nem em desacordo": 3, "Mais ou menos de acordo": 4, "De acordo": 5, "Totalmente de acordo": 6}
IN50_54 = [f"IN{i}" for i in range(50, 55)]
ordinal(IN50_54, CONCORDANCIA)
registar(IN50_54, "IN: opiniões", "Grau de concordância", "Ordinal 0-6; 'Sem resposta' → NaN", "Escala ordenada de 7 pontos")

### 3.10 Mobilidade: `IN10`–`IN13`
`IN11`/`IN12` só são perguntados a quem respondeu "Sim" em `IN10`. `IN11` tem uma terceira resposta
("Desloca-se em cadeira de rodas") → categórica. `IN13` tem 99,9% de "Não aplicável" (11 respostas) → removida.

In [30]:
cruzar("IN11", "IN10")
ver("IN13")
ordinal(["IN10", "IN12"], SIM_NAO)
categorica(["IN11"])
X = X.drop(columns="IN13")
registar(["IN10", "IN12"], "IN: mobilidade", "Perguntas Sim/Não de mobilidade", "Binária 1/0", "'Não aplicável' explicado por IN10")
registar("IN11", "IN: mobilidade", "Mobilidade (Sim / Não / cadeira de rodas)", "Categórica nominal", "Terceira resposta sem ordem clara")
registar("IN13", "IN: mobilidade", "Deslocação (11 respostas)", "Removida", "99,9% 'Não aplicável'")

IN10,Não,Não aplicável,Sem resposta,Sim
IN11,,,,
Desloca-se em cadeira de rodas,0,0,0,26
Não,0,0,3,371
Não aplicável,2613,11472,0,0
Sem resposta,0,0,6,5
Sim,0,0,0,121


=== IN13  (object) | 4 valores distintos | V de Cramér com a idade = 0.041


,treino,teste,faixa_media
IN13,,,
Não aplicável,10955,3651.0,8.8
Só com ajuda,3,3.0,14.7
"Sozinho, mas com dificuldade",2,0.0,11.0
"Sozinho, sem dificuldade",2,1.0,12.5


### 3.11 Ausência ao trabalho, quantidades e `IN67`

In [31]:
categorica(["IN14"])
numerica(["IN55", "IN56", "IN57", "IN59"])
ordinal(["IN67"], SIM_NAO)
registar("IN14", "IN: outros", "Motivo de ausência", "Categórica nominal (+ indicadora de 'Sem resposta')", "Sem ordem natural")
registar(["IN55", "IN56", "IN57", "IN59"], "IN: outros", "Quantidades declaradas", "Numérica (+ indicadora de 'Sem resposta' se ≥ 1%)",
         "Já são números")
registar("IN67", "IN: outros", "Pergunta Sim/Não", "Binária 1/0", "Duas respostas")

restam = [c for c in X.columns if c.startswith("IN") and X[c].dtype == object]
assert not restam, restam
print("colunas IN tratadas; indicadoras criadas:", [c for c in X.columns if c.endswith(("_SEM_RESP", "_SEM_TERMO"))])
print(X.shape)

colunas IN tratadas; indicadoras criadas: ['IN7a_SEM_RESP', 'IN7b_SEM_RESP', 'IN7c_SEM_RESP', 'IN7d_SEM_RESP', 'IN7e_SEM_RESP', 'IN7f_SEM_RESP', 'IN7g_SEM_RESP', 'IN7h_SEM_RESP', 'IN24_GRAVIDEZ_SEM_TERMO', 'IN26_SEM_RESP', 'IN40_SEM_RESP', 'IN45_SEM_RESP', 'IN47_SEM_RESP', 'IN14_SEM_RESP', 'IN55_SEM_RESP', 'IN56_SEM_RESP', 'IN57_SEM_RESP', 'IN59_SEM_RESP']
(14617, 245)


## 4. Restantes módulos: escolaridade, trabalho, rendimento e módulos de saúde

Aqui o tratamento é **só de codificação** (texto → números), com a mesma regra geral do bloco 3:
- escalas com ordem → ordinais pela ordem natural;
- Sim/Não → 1/0;
- números em texto → número (limites como "≥ 25" → 25; `PE7`/`PE9` estão em HHMM → minutos);
- respostas sem ordem → categóricas;
- respostas especiais que são informação ("nunca tentou", "deixou de beber", "não consegue avaliar")
  → indicadora própria; "Não aplicável" / "Sem resposta" / "NR" → `NaN`.

In [32]:
SAUDE = {"Muito mau": 0, "Mau": 1, "Razoável": 2, "Bom": 3, "Muito bom": 4}
MENTAL = {"Nunca": 0, "Vários dias": 1, "Mais do que metade dos dias": 2, "Quase todos os dias": 3}
DURACAO = {"10 a 29 minutos": 0, "30 a 59 minutos": 1, "1 hora a menos de 2 horas": 2,
           "2 horas a menos de 3 horas": 3, "3 horas ou mais": 4}
DIAS_ALCOOL = {"Em nenhum dos dias": 0, "1 dia": 1, "2 dias": 2, "3 dias": 3, "4 dias": 4}
BEBIDAS = {"0 bebidas por dia": 0, "1 bebida por dia": 1, "2 bebidas por dia": 2, "3 bebidas por dia": 3,
           "4 a 5 bebidas por dia": 4, "6 a 9 bebidas por dia": 5, "10 a 15 bebidas por dia": 6, "16 ou mais bebidas por dia": 7}
FREQ_ALCOOL = {"Nunca consumiu, ou só ocasionalmente para provar": 0,
               "Não consumiu nos últimos 12 meses, por ter deixado de consumir álcool": 0,
               "Menos de uma vez por mês": 1, "Uma vez por mês": 2, "2 a 3 dias por mês": 3, "1 a 2 dias por semana": 4,
               "3 a 4 dias por semana": 5, "5 a 6 dias por semana": 6, "Todos os dias ou quase todos os dias": 7}
PA_3_5 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 3 anos": 1, "Entre 3 anos e menos de 5 anos": 2, "5 anos ou mais": 3, "Nunca": 4}
PA_2_3 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 2 anos": 1, "Entre 2 anos e menos de 3 anos": 2, "3 anos ou mais": 3, "Nunca": 4}
PA_5_10 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 5 anos": 1, "Entre 5 anos e menos de 10 anos": 2, "10 anos ou mais": 3, "Nunca": 4}
PA_2_3_5 = {"Nos últimos 12 meses": 0, "Entre 1 ano e menos de 2 anos": 1, "Entre 2 anos e menos de 3 anos": 2,
            "Entre 3 anos e menos de 5 anos": 3, "5 anos ou mais": 4, "Nunca": 5}

ESCALAS = {
    "HATLEVEL": {"Nenhum nível completado": 0, "Básico 1º e 2º ciclo": 1, "Básico 3º ciclo": 2, "Secundário": 3,
                 "Pós secundário": 4, "C.T.Superior Profissional": 5, "Bacharelato ou licenciatura 1º ciclo": 6,
                 "Licenciatura pré-bolonha ou mestrado 2º ciclo": 7, "Doutoramento": 8},
    "HHINCOME": {"1º quintil": 1, "2º quintil": 2, "3º quintil": 3, "4º quintil": 4, "5º quintil": 5},
    "FT_PT": {"Tempo parcial": 0, "Tempo completo": 1},
    "HS1": SAUDE, "CD2": SAUDE,
    "HS3": {"Nada limitado": 0, "Limitado mas não severamente": 1, "Severamente limitado": 2},
    "AW2_COD": {"1-7": 0, "8-14": 1, "15-30": 2, "31-180": 3, ">= 181": 4},
    "PL1": {"Não": 0, "Sim": 1, "Cego (não consegue ver de todo)": 2},
    "PL3": {"Não": 0, "Sim": 1, "Surdo (surdez profunda)": 2},
    **{c: DIFICULDADE for c in ["PL2", "PL4", "PL5", "PL6", "PL7", "PL8", "PL9_tot",
                                "PC1A", "PC1B", "PC1C", "PC1D", "PC1E", "HA1A", "HA1B", "HA1C", "HA1D", "HA1E", "HA1F", "HA1G"]},
    "PN1": {"Nenhuma": 0, "Muito ligeira": 1, "Ligeira": 2, "Moderada": 3, "Intensa": 4, "Muito intensa": 5},
    "PN2": {"Nada": 0, "Um pouco": 1, "De forma moderada": 2, "Bastante": 3, "De forma extrema": 4},
    **{f"MH1{l}": MENTAL for l in "ABCDEFGH"},
    "AM1": {"Há menos de 6 meses": 0, "6 a menos de 12 meses": 1, "Há 12 meses ou mais": 2, "Nunca": 3},
    "AM2": {"Há menos de 12 meses": 0, "Há 12 meses ou mais": 1, "Nunca": 2},
    "AM4": {"Há menos de 12 meses": 0, "Há 12 meses ou mais": 1, "Nunca": 2},
    "PA2": PA_3_5, "PA3": PA_3_5, "PA4": PA_3_5, "PA5": PA_2_3, "PA6": PA_5_10, "PA7_NAC": PA_2_3_5, "PA8_NAC": PA_2_3_5,
    "UN2A": None, "UN2B": None, "UN2C": None, "UN2D": None,   # tratadas abaixo (com 'não houve necessidade')
    "PE1": {"Não executa nenhuma das tarefas indicadas": 0,
            "Sentada ou em pé em atividades que envolvem um esforço físico ligeiro": 1,
            "Em movimento ou em tarefas que exigem um esforço físico moderado": 2,
            "Em trabalhos pesados ou fisicamente exigentes": 3},
    "PE3": DURACAO, "PE5": DURACAO,
    **{c: FREQ for c in ["DH1", "DH3", "DH5", "DH6"]},
    "SK5": {"Nunca ou quase nunca": 0, "Menos de uma vez por semana": 1,
            "Pelo menos uma vez por semana (mas não diariamente)": 2,
            "Diariamente, menos de 1 hora por dia": 3, "Diariamente, 1 hora ou mais por dia": 4},
    "SK6": {"Nunca": 0, "Não, mas já utilizou": 1, "Sim, ocasionalmente": 2, "Sim, diariamente": 3},
    "AL1": FREQ_ALCOOL, "AL6": FREQ_ALCOOL, "AL2": DIAS_ALCOOL, "AL4": DIAS_ALCOOL, "AL3": BEBIDAS, "AL5": BEBIDAS,
    "SS1": {"Nenhuma": 0, "1 ou 2": 1, "3 a 5": 2, "6 ou mais": 3},
    "SS2": {"Nenhuma preocupação e interesse": 0, "Pouca preocupação e interesse": 1,
            "Alguma preocupação e interesse": 2, "Muita preocupação e interesse": 3},
    "SS3": {"Muito difícil": 0, "Difícil": 1, "Possível": 2, "Fácil": 3, "Muito fácil": 4},
    "IC3": {"Menos de 10 horas": 0, "Entre 10 horas e menos de 20 horas": 1, "20 horas ou mais": 2},
}
SIM_NAO_COLS = ["HS2"] + [f"CD1{l}" for l in "ABCDEFGHIJKLMNOP"] + ["AC1A", "AC1B", "AC1C", "AC_IN1", "AC_IN2",
                "AW1", "PC2", "PC3", "HA2", "HA3", "AM6A", "AM6B", "AM7", "MD1", "MD2", "NT1", "NT2", "NT3", "SK3", "IC1"]
CATEGORICAS = ["MAINSTAT", "JOBSTAT_COD", "JOBISCO_COD", "LOCNACE_COD", "AC2", "UN1A", "UN1B",
               "UN2A", "UN2B", "UN2C", "UN2D", "IC2"]
NUMERICAS = ["HO12_COD", "HO34_COD", "AM3", "AM5", "BM1", "BM2", "PE2", "PE4", "PE6", "PE8", "DH2", "DH4", "SK2"]
LIMITES = {"HO12_COD": {">= 25": "25"}, "HO34_COD": {">= 60": "60"}}
ESPECIAIS_INFO = {  # resposta especial -> sufixo da indicadora
    "Não aplicável (nunca tentou ou não precisa de o fazer)": "NUNCA_TENTOU",
    "Não consumiu nos últimos 12 meses, por ter deixado de consumir álcool": "DEIXOU_ALCOOL",
    "Não consegue avaliar": "NAO_CONSEGUE_AVALIAR",
}

In [33]:
# 1) indicadoras das respostas especiais que são informação (antes de mapear)
novas = {}
for c in [c for c in ESCALAS if ESCALAS[c] is not None]:
    for texto, sufixo in ESPECIAIS_INFO.items():
        if X[c].eq(texto).any():
            novas[f"{c}_{sufixo}"] = X[c].eq(texto).astype(int)
X = pd.concat([X, pd.DataFrame(novas)], axis=1)
print("indicadoras de respostas especiais:", list(novas))

# 2) escalas, Sim/Não, números, categóricas
for c, mapa in ESCALAS.items():
    if mapa is not None:
        ordinal([c], {**mapa, **{t: np.nan for t in ESPECIAIS_INFO if t not in mapa}})
ordinal(SIM_NAO_COLS, SIM_NAO)
numerica(NUMERICAS, {**LIMITES.get("HO12_COD", {}), **LIMITES.get("HO34_COD", {})})
categorica(CATEGORICAS)

# 3) PE7 / PE9 estão em HHMM ("0130" = 1h30) -> minutos
for c in ["PE7", "PE9"]:
    hhmm = pd.to_numeric(X[c].astype(str), errors="coerce")
    sem_resposta(c)
    X[c] = (hhmm // 100) * 60 + hhmm % 100

# 4) PA1: data (AAAAMM ou só AAAA) -> meses desde essa data até à entrevista; texto ("Nunca", ...) fica categórico
pa1 = pd.to_numeric(X["PA1"].astype(str), errors="coerce")
ano = np.where(pa1 > 9999, pa1 // 100, pa1)
mes = np.where(pa1 > 9999, pa1 % 100, 6)                       # só o ano -> meio do ano
entrevista = pd.concat([train["REFYEAR"] * 12 + train["REFMONTH"], test["REFYEAR"] * 12 + test["REFMONTH"]]).values
X["PA1_MESES"] = entrevista - (ano * 12 + mes)
X["PA1"] = X["PA1"].astype(str).where(pa1.isna(), "Data indicada")
categorica(["PA1"])

restam = [c for c in X.columns if X[c].dtype == object]
assert not restam, restam
print(X.shape)
print(X.dtypes.map(lambda t: t.name).value_counts())

indicadoras de respostas especiais: ['HA1A_NUNCA_TENTOU', 'HA1B_NUNCA_TENTOU', 'HA1C_NUNCA_TENTOU', 'HA1D_NUNCA_TENTOU', 'HA1E_NUNCA_TENTOU', 'HA1F_NUNCA_TENTOU', 'HA1G_NUNCA_TENTOU', 'AL1_DEIXOU_ALCOOL', 'AL6_DEIXOU_ALCOOL', 'SS2_NAO_CONSEGUE_AVALIAR']


(14617, 261)
float64     196
int64        35
category     30
Name: count, dtype: int64


In [34]:
registar("HATLEVEL", "Escolaridade", "Nível de escolaridade", "Ordinal 0-8", "Ordem natural dos níveis de ensino")
registar("HHINCOME", "Rendimento", "Quintil de rendimento do agregado", "Ordinal 1-5", "Ordem natural")
registar("FT_PT", "Trabalho", "Tempo parcial / completo", "Binária (1 = completo)", "Duas respostas; 'Não aplicável' = não trabalha")
registar(["MAINSTAT", "JOBSTAT_COD", "JOBISCO_COD", "LOCNACE_COD"], "Trabalho", "Situação, vínculo, profissão, setor",
         "Categórica nominal", "Sem ordem natural")
registar(["HS1", "CD2"], "Saúde", "Autoavaliação do estado de saúde", "Ordinal 0-4", "Muito mau < ... < Muito bom")
registar("HS3", "Saúde", "Limitação por problemas de saúde", "Ordinal 0-2", "Ordem natural")
registar(SIM_NAO_COLS, "Saúde", "Perguntas Sim/Não (doenças crónicas, acidentes, cuidados, medicamentos, ...)",
         "Binária 1/0; especiais → NaN", "Duas respostas")
registar(["PL1", "PL3"], "Limitações", "Usa óculos / aparelho auditivo", "Ordinal (Não 0, Sim 1, cego/surdo 2)", "Cego/surdo é o grau mais severo")
registar([c for c in ESCALAS if c.startswith(("PL", "PC1", "HA1"))], "Limitações", "Grau de dificuldade",
         "Ordinal 0-3 (+ indicadora 'nunca tentou' em HA1*)", "Ordem natural; PC1*/HA1* só perguntados a partir dos 55 anos")
registar(["PN1", "PN2"], "Dor", "Intensidade / interferência da dor", "Ordinal", "Ordem natural")
registar([f"MH1{l}" for l in "ABCDEFGH"], "Saúde mental", "Frequência de sintomas (PHQ-8)", "Ordinal 0-3", "Escala PHQ-8")
registar(["AM1", "AM2", "AM4", "PA2", "PA3", "PA4", "PA5", "PA6", "PA7_NAC", "PA8_NAC"], "Cuidados / prevenção",
         "Há quanto tempo (escalões)", "Ordinal (Nunca = último valor)", "Ordem natural do tempo")
registar(["PA1"], "Prevenção", "Data do último exame (ou texto)", "Categórica ('Data indicada' / texto)", "Separa data de respostas em texto")
registar("PA1_MESES", "Prevenção", "Meses desde a data de PA1 até à entrevista (nova)", "Numérica",
         "Uma data em si não é comparável; a distância à entrevista é")
registar(["AC2", "UN1A", "UN1B", "UN2A", "UN2B", "UN2C", "UN2D", "IC2"], "Saúde", "Tipo de cuidado / necessidades não satisfeitas / cuidador",
         "Categórica nominal", "'Não houve necessidade' não cabe numa escala Sim/Não")
registar(NUMERICAS, "Vários", "Quantidades (consultas, altura, peso, dias, cigarros, ...)", "Numérica ('≥ 25' → 25, '≥ 60' → 60)", "Já são números")
registar(["PE7", "PE9"], "Atividade física", "Tempo (HHMM)", "Numérica em minutos", "'0130' = 90 minutos")
registar(["PE1", "PE3", "PE5"], "Atividade física", "Esforço no trabalho / duração", "Ordinal", "Ordem natural")
registar(["DH1", "DH3", "DH5", "DH6"], "Alimentação", "Frequência de consumo", "Ordinal 0-4", "Ordem natural")
registar(["SK5", "SK6"], "Tabaco", "Exposição ao fumo / cigarro eletrónico", "Ordinal", "Ordem de exposição")
registar(["AL1", "AL6", "AL2", "AL4", "AL3", "AL5"], "Álcool", "Frequência / dias / bebidas", "Ordinal (+ indicadora 'deixou de beber')",
         "Ordem natural; 'deixou de beber' = 0 com indicadora própria")
registar(["SS1", "SS2", "SS3"], "Apoio social", "Escala de apoio social", "Ordinal (+ indicadora 'não consegue avaliar' em SS2)", "Ordem natural")
registar("IC3", "Cuidados informais", "Horas de cuidados", "Ordinal 0-2", "Ordem natural")

## 5. Gravar

As categorias do teste são alinhadas com as do **treino** (uma categoria que só exista no teste passa a `NaN`),
para o modelo nunca ver no teste um código que não aprendeu.

In [35]:
train_v2 = X.loc["treino"].copy()
test_v2 = X.loc["teste"].copy()
for c in train_v2.select_dtypes("category").columns:
    train_v2[c] = train_v2[c].cat.remove_unused_categories()
    test_v2[c] = pd.Categorical(test_v2[c], categories=train_v2[c].cat.categories)
train_v2["AGE_RANGE"] = y.values

assert len(test_v2) == len(test) and list(test_v2.index) == list(range(len(test)))   # ordem do teste intacta
train_v2.reset_index(drop=True).to_parquet("../dados/train_tratado_v2.parquet")
test_v2.reset_index(drop=True).to_parquet("../dados/test_tratado_v2.parquet")
pd.DataFrame(decisoes).to_csv("../resultados/decisoes_tratamento.csv", index=False, encoding="utf-8-sig")
print("treino", train_v2.shape, "| teste", test_v2.shape, "| decisões registadas:", len(decisoes))

treino (10962, 262) | teste (3655, 261) | decisões registadas: 240
